# Klasifikasi DemogPairs Menggunakan ViT (Wajah, Emosi, dan Umur) & Random Forest

In [1]:
import numpy as np
import utils as u
import joblib
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, ParameterGrid
from imblearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from tqdm import tqdm

joblib.parallel_backend('threading')

## Load Dataset

In [2]:
data = u.load_demogpairs()
pd.DataFrame(data)

,db_code,image_path,full_path,label,label_idx
0,CWF,able_wanamakok/002.jpg,dataset/demogpairs/images\able_wanamakok/002.jpg,Asian_Females,5
1,CWF,able_wanamakok/004.jpg,dataset/demogpairs/images\able_wanamakok/004.jpg,Asian_Females,5
2,CWF,able_wanamakok/007.jpg,dataset/demogpairs/images\able_wanamakok/007.jpg,Asian_Females,5
3,CWF,able_wanamakok/008.jpg,dataset/demogpairs/images\able_wanamakok/008.jpg,Asian_Females,5
4,CWF,able_wanamakok/012.jpg,dataset/demogpairs/images\able_wanamakok/012.jpg,Asian_Females,5
...,...,...,...,...,...
10795,CWF,zachary_quinto/177.jpg,dataset/demogpairs/images\zachary_quinto/177.jpg,White_Males,3
10796,CWF,zachary_quinto/214.jpg,dataset/demogpairs/images\zachary_quinto/214.jpg,White_Males,3
10797,CWF,zachary_quinto/217.jpg,dataset/demogpairs/images\zachary_quinto/217.jpg,White_Males,3
10798,CWF,zachary_quinto/218.jpg,dataset/demogpairs/images\zachary_quinto/218.jpg,White_Males,3


## Load Fitur

In [3]:
face_features = joblib.load('features/demogpairs_vit-face.pkl')
emotion_features = joblib.load('features/demogpairs_vit-emotion.pkl')
age_features = joblib.load('features/demogpairs_vit-age.pkl')
features = {}
for d in tqdm(data):
    key = d['image_path']
    features[key] = np.array(list(face_features[key]) + list(emotion_features[key]) + list(age_features[key]))
print('Jumlah fitur per gambar:', np.array(features[list(features.keys())[0]]).shape[0])

100%|██████████████████████████████████████████████████████████████████████████| 10800/10800 [00:01<00:00, 7289.40it/s]

Jumlah fitur per gambar: 2304


## Split Data

In [4]:
X = np.array([features[d['image_path']] for d in data])
y = np.array([d['label_idx'] for d in data])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print((len(X_train), len(X_test)))

(8640, 2160)


## Kombinasi Parameter

In [5]:
grid_params = [
    {
        'scaler': [None, MinMaxScaler()],
        'pca': [None, PCA(n_components=0.5), PCA(n_components=0.75)],
        
        'classifier': [RandomForestClassifier(random_state=42)],
        'classifier__n_estimators': [100, 200],
        'classifier__max_depth': [None, 20, 30],
        'classifier__min_samples_split': [2, 5],
        'classifier__min_samples_leaf': [1, 2],
        'classifier__max_features': ['sqrt', 'log2'],
    },
]

pipeline = Pipeline(steps=[
    ('scaler', None),
    ('pca', None),
    ('classifier', None)
])

skv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    'accuracy': 'accuracy', 
    'f1': 'f1_macro', 
    'precision': 'precision_macro', 
    'recall': 'recall_macro',
}

grid_models = {}
for params in grid_params:
    key = str(params['classifier'][0]).split('(')[0]
    grid_models[key] = GridSearchCV(
        estimator=pipeline,
        param_grid=params,
        cv=skv, refit='accuracy',
        scoring=scoring, n_jobs=int(joblib.cpu_count() * 0.6),
        verbose=1, error_score='raise',
        return_train_score=True
    )
    print(f'{key}: {len(ParameterGrid(params))} kombinasi')

RandomForestClassifier: 288 kombinasi


## Klasifikasi

In [6]:
evaluation_results, fold_results = u.evaluate_models(
    grid_models, 
    X_train, y_train,
    X_test, y_test,
    target_names=u.demogpairs_classes,
    model_prefix="models/clf_demogpairs_rf_vit-face-emotion-age_",
    results_path="results/demogpairs_rf_vit-face-emotion-age_"
)
sorted_results = pd.DataFrame(evaluation_results).sort_values(by="test_accuracy", ascending=False).to_dict("records")
u.html_br()
_dtable = u.display_table(sorted_results)

Evaluating: RandomForestClassifier


{'classifier': 'RandomForestClassifier', 'classifier__max_depth': 30, 'classifier__max_features': 'sqrt', 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 200, 'pca': 'PCA', 'scaler': None}


Accuracy  : 0.862037037037037
Precision : 0.8619831983316151
Recall    : 0.8620370370370369
F1 Score  : 0.8613389248830209
               precision    recall  f1-score   support

Asian_Females     0.8850    0.9194    0.9019       360
  Asian_Males     0.8683    0.8972    0.8825       360
Black_Females     0.8264    0.8333    0.8299       360
  Black_Males     0.8714    0.9222    0.8961       360
White_Females     0.8793    0.7889    0.8316       360
  White_Males     0.8415    0.8111    0.8260       360

     accuracy                         0.8620      2160
    macro avg     0.8620    0.8620    0.8613      2160
 weighted avg     0.8620    0.8620    0.8613      2160



Class,OvR Accuracy,Precision,Recall,F1-Score,Support
Asian_Females,0.9666666666666667,0.8850267379679144,0.9194444444444444,0.9019073569482288,360
Asian_Males,0.9601851851851851,0.8682795698924731,0.8972222222222223,0.8825136612021858,360
Black_Females,0.9430555555555555,0.8264462809917356,0.8333333333333334,0.8298755186721992,360
Black_Males,0.9643518518518519,0.8713910761154856,0.9222222222222223,0.8960863697705803,360
White_Females,0.9467592592592593,0.8792569659442725,0.7888888888888889,0.8316251830161054,360
White_Males,0.9430555555555555,0.8414985590778098,0.8111111111111111,0.8260254596888261,360


Confusion matrix saved: images\cm_rf_vit-face-emotion-age_RandomForestClassifier.png



Confusion Matrix:
                         Asian_Females       Asian_Males     Black_Females       Black_Males     White_Females       White_Males
       Asian_Females               331                 0                10                13                 6                 0
         Asian_Males                 2               323                 4                 2                11                18
       Black_Females                 8                 1               300                28                 1                22
         Black_Males                 8                 6                12               332                 0                 2
       White_Females                25                23                11                 4               284                13
         White_Males                 0                19                26                 2                21               292


model_name,model_file_path,best_parameters,test_accuracy,test_f1,test_precision,test_recall,parameter_combinations
RandomForestClassifier,models/clf_demogpairs_rf_vit-face-emotion-age_RandomForestClassifier.pkl,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': 30, 'classifier__max_features': 'sqrt', 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 200, 'pca': 'PCA', 'scaler': None}",0.862037037037037,0.8613389248830209,0.8619831983316151,0.8620370370370369,288


In [7]:
model, training_time = u.load_object('models/clf_demogpairs_rf_vit-face-emotion-age_RandomForestClassifier.pkl')
u.h(5, 'Waktu Pelatihan (Jobs)')
u.seconds_to_time(round(training_time))

{'input_seconds': 9553.0,
 'days': 0,
 'hours': 2,
 'minutes': 39,
 'seconds': 13.0,
 'text': '0 hari 2 jam 39 menit 13.0 detik'}

In [8]:
u.h(5, 'Waktu Pelatihan')
times = [fr['Train Time Mean'] * 5 for fr in fold_results]
u.seconds_to_time(round(np.sum(times) + model.refit_time_))

{'input_seconds': 37194.0,
 'days': 0,
 'hours': 10,
 'minutes': 19,
 'seconds': 54.0,
 'text': '0 hari 10 jam 19 menit 54.0 detik'}

In [9]:
_dtable = u.display_table(fold_results, n_items=[4, 4], column_widths=['5%', '45%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%'])

No,Params,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5,Accuracy Mean,F1 Score Mean,Precision Mean,Recall Mean,Train Time Mean
1,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': 30, 'classifier__max_features': 'sqrt', 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 200, 'pca': 'PCA', 'scaler': None}",0.886,0.8733,0.8623,0.8779,0.8744,0.8748,0.8744,0.875,0.8748,30.2795
2,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': None, 'classifier__max_features': 'sqrt', 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 200, 'pca': 'PCA', 'scaler': None}",0.8854,0.8727,0.8623,0.8785,0.8733,0.8744,0.874,0.8747,0.8744,29.9262
3,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': 20, 'classifier__max_features': 'sqrt', 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 200, 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.8819,0.8681,0.8657,0.8808,0.8721,0.8737,0.8733,0.874,0.8737,30.6476
4,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': 20, 'classifier__max_features': 'sqrt', 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 200, 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.8785,0.8709,0.8646,0.8831,0.8709,0.8736,0.8732,0.874,0.8736,29.5666
...,...,...,...,...,...,...,...,...,...,...,...
285,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': None, 'classifier__max_features': 'log2', 'classifier__min_samples_leaf': 2, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 100, 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.8704,0.8495,0.8605,0.8617,0.8663,0.8617,0.8611,0.862,0.8617,16.939
286,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': 30, 'classifier__max_features': 'log2', 'classifier__min_samples_leaf': 2, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 100, 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.8698,0.8495,0.86,0.8617,0.8657,0.8613,0.8608,0.8616,0.8613,18.5868
287,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': 20, 'classifier__max_features': 'log2', 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 100, 'pca': None, 'scaler': 'MinMaxScaler'}",0.8796,0.8565,0.8501,0.8565,0.8623,0.861,0.8602,0.8615,0.861,12.2542
288,"{'classifier': 'RandomForestClassifier', 'classifier__max_depth': 20, 'classifier__max_features': 'log2', 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 100, 'pca': None, 'scaler': None}",0.8796,0.8565,0.8501,0.8565,0.8623,0.861,0.8602,0.8615,0.861,7.4332
